# Datenvorbereitung

In [4]:
import pandas as pd
import numpy as np
import os

def process_healthcare_data():
    """
    Reads, cleans, and merges healthcare supply data (KBV) and population data (INKAR).
    Designed to run from the 'notebooks/' directory, accessing data in '../data/'.
    Output: A combined CSV file saved in the '../data/' directory.
    """
    
    # ---------------------------------------------------------
    # 1. Configuration & File Paths
    # ---------------------------------------------------------
    # ".." means: Go one level up (exit 'notebooks' folder)
    # "data" means: Enter 'data' folder
    INPUT_FOLDER = os.path.join('..', 'data')
    OUTPUT_FILENAME = 'merged_data_therapy.csv'
    
    # Construct full paths
    file_path_inkar = os.path.join(INPUT_FOLDER, 'InkarDaten.csv')
    file_path_kbv = os.path.join(INPUT_FOLDER, 'KBV_Daten.csv')
    output_path = os.path.join(INPUT_FOLDER, OUTPUT_FILENAME)

    # Print current working directory for debugging clarity
    print(f"Current Working Directory: {os.getcwd()}")
    print(f"Looking for data in: {os.path.abspath(INPUT_FOLDER)}")

    # Verify files exist before proceeding
    if not os.path.exists(file_path_inkar):
        raise FileNotFoundError(f"File not found: {file_path_inkar} (Check if your script is actually running inside the 'notebooks' folder)")
    if not os.path.exists(file_path_kbv):
        raise FileNotFoundError(f"File not found: {file_path_kbv}")

    print("Datasets found. Starting processing...")

    # ---------------------------------------------------------
    # 2. Process INKAR Data (Population Density)
    # ---------------------------------------------------------
    # 'Kennziffer' is the AGS (Amtlicher Gemeindeschlüssel). 
    # Read as string to preserve leading zeros.
    df_inkar = pd.read_csv(
        file_path_inkar, 
        sep=';', 
        dtype={'Kennziffer': str}, 
        encoding='utf-8'
    )

    # Clean: Remove metadata rows (often the first row contains year info like '2023')
    if pd.isna(df_inkar.iloc[0]['Raumeinheit']) or str(df_inkar.iloc[0]['Raumeinheit']).isdigit():
        df_inkar = df_inkar.drop(0)

    # Rename AGS column for consistency
    df_inkar = df_inkar.rename(columns={'Kennziffer': 'AGS'})

    # Helper function for German number format: '1.234,56' -> 1234.56
    def convert_german_float(value):
        if pd.isna(value):
            return np.nan
        if isinstance(value, str):
            value = value.replace('.', '')  # Remove thousands separator
            value = value.replace(',', '.')  # Replace decimal separator
        return float(value)

    # Convert Population Density
    df_inkar['Einwohnerdichte'] = df_inkar['Einwohnerdichte'].apply(convert_german_float)

    # Select only necessary columns
    df_inkar_subset = df_inkar[['AGS', 'Raumeinheit', 'Einwohnerdichte']]

    # ---------------------------------------------------------
    # 3. Process KBV Data (Provision & Waiting Times)
    # ---------------------------------------------------------
    df_kbv = pd.read_csv(
        file_path_kbv, 
        dtype={'AGS': str}, 
        encoding='utf-8'
    )

    # Filter invalid rows (Excel '#NV' errors or empty AGS)
    df_kbv = df_kbv[df_kbv['AGS'].notna()]
    df_kbv = df_kbv[df_kbv['AGS'] != '#NV']

    # Convert columns to numeric
    df_kbv['Versorgungsgrad'] = df_kbv['Versorgungsgrad'].apply(convert_german_float)
    
    # 'Wartezeit genau' represents median days. Force numeric conversion.
    df_kbv['Wartezeit genau'] = pd.to_numeric(df_kbv['Wartezeit genau'], errors='coerce')

    # Select only necessary columns
    df_kbv_subset = df_kbv[['AGS', 'Versorgungsgrad', 'Wartezeit genau']]

    # ---------------------------------------------------------
    # 4. Merge Datasets
    # ---------------------------------------------------------
    # Inner Join ensures we only keep regions present in both datasets
    df_merged = pd.merge(df_inkar_subset, df_kbv_subset, on='AGS', how='inner')

    # ---------------------------------------------------------
    # 5. Final Formatting
    # ---------------------------------------------------------
    # Rename columns to clean, descriptive English names
    final_columns = {
        'AGS': 'AGS',
        'Raumeinheit': 'Region_Name',
        'Einwohnerdichte': 'Population_Density',
        'Versorgungsgrad': 'Provision_Rate_Percent',
        'Wartezeit genau': 'Waiting_Time_Days_Median'
    }
    
    df_final = df_merged.rename(columns=final_columns)

    # Sort by AGS for logical order
    df_final = df_final.sort_values(by='AGS').reset_index(drop=True)

    # Save to CSV in the data folder
    df_final.to_csv(output_path, sep=';', index=False, float_format='%.2f')
    
    print(f"Success! Processed {len(df_final)} regions.")
    print(f"File saved to: {output_path}")
    print("\nData Preview:")
    print(df_final.head())

if __name__ == "__main__":
    try:
        process_healthcare_data()
    except Exception as e:
        print(f"An error occurred: {e}")

Current Working Directory: /Users/Arina/Documents/TH/Datenvisualisierung/datenvisualisierung-/notebooks
Looking for data in: /Users/Arina/Documents/TH/Datenvisualisierung/datenvisualisierung-/data
Datasets found. Starting processing...
Success! Processed 47 regions.
File saved to: ../data/merged_data_therapy.csv

Data Preview:
     AGS        Region_Name  Population_Density  Provision_Rate_Percent  \
0  09361             Amberg              841.98                  110.60   
1  09362  Regensburg, Stadt             1850.90                  184.65   
2  09363    Weiden i.d.OPf.              601.25                  107.07   
3  09371    Amberg-Sulzbach               82.91                  110.60   
4  09372               Cham               83.98                  112.94   

   Waiting_Time_Days_Median  
0                     103.0  
1                      91.0  
2                     169.0  
3                     103.0  
4                      94.0  
